# Practical 05 — Hyperspectral Satellite Imaging & Spectroscopy
**PLNT 7620 · Part 4: Satellite Remote Sensing · University of Manitoba · Digital AgroEcosystems Lab**

**Learning goals:** Load authentic shared hyperspectral/spectroscopy data; distinguish spectra from satellite images; evaluate wavelength coverage and quality; inspect soil/vegetation signatures; compare raw vs preprocessed spectra; explore multivariate structure (PCA); optionally fit a simple prediction model only when labelled targets exist.

**Data source:** [Instructor-provided Drive folder](https://drive.google.com/drive/folders/1MZeno2uHjEn-hFJTYknID5yaUFFbFjPE?usp=sharing). Raw input remains outside this public GitHub repository. Outputs save to **your own** Google Drive.

**Start:** File → Save a copy in Drive; run cells in order. Google Drive mounting needs consent. The instructor's folder must be shared so your Google account can access it. If automatic download fails, manually add a *shortcut* to that shared folder in My Drive or upload its files to Colab; the notebook never requires the instructor's credentials.

**Scientific caution:** A point spectrometer captures spectral signatures at sampling spots. An imaging spectrometer records spectra in spatial pixels. Absorption features and reflectance shapes depend on sensor characteristics, atmospheric correction, moisture, illumination and sample preparation; they are not direct chemical measurements.

In [ ]:
%pip -q install gdown openpyxl scipy scikit-learn pandas matplotlib

## 1. Connect your Google Drive and instructor data folder
The first connection mounts **your Drive** for reproducible exports. Independently, `gdown` attempts to read the **shared instructor folder**. If folder access is restricted, copy relevant CSV/XLSX files into your own input folder and rerun the discovery cell.

In [ ]:
from pathlib import Path
import re, shutil, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

SHARED_FOLDER_URL = "https://drive.google.com/drive/folders/1MZeno2uHjEn-hFJTYknID5yaUFFbFjPE?usp=sharing"
STUDENT_ID = "student"  # optional: replace with non-sensitive identifier
WORK_DIR = Path("/content/Practical_05_Hyperspectral")
INPUT_DIR = WORK_DIR / "instructor_data"
INPUT_DIR.mkdir(parents=True, exist_ok=True)

try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive")
except (ImportError, RuntimeError, ValueError) as e:
    print("Drive not mounted:", e)
    DRIVE_ROOT = WORK_DIR
OUTPUT_DIR = DRIVE_ROOT / "PLNT7620_Part4" / "Practical_05" / re.sub(r"[^a-zA-Z0-9_-]", "_", STUDENT_ID)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Outputs will be saved to:", OUTPUT_DIR)

In [ ]:
import gdown
try:
    downloaded = gdown.download_folder(url=SHARED_FOLDER_URL, output=str(INPUT_DIR), quiet=False, remaining_ok=True)
    print("Download returned", len(downloaded or []), "files.")
except Exception as e:
    print("Shared-folder download was not completed:", type(e).__name__, str(e)[:500])
    print("Open the instructor folder, verify permissions, and place CSV/XLSX data in:", INPUT_DIR)
print("Local inputs:", [str(p.relative_to(INPUT_DIR)) for p in INPUT_DIR.rglob("*") if p.is_file()][:40])

## 2. Discover spectral tables and identify the layout
The importer supports two common layouts: **one spectrum per row with wavelength-labelled headers**, and **one wavelength per row with sample spectra in columns**. Select the correct file index below; other shared files may contain metadata or imagery rather than tabular reflectance. Wavelengths must be explicitly present (350–2500 nm); the notebook will not invent wavelengths for unnamed bands.


In [ ]:
files = sorted([p for p in INPUT_DIR.rglob("*") if p.suffix.lower() in (".csv", ".tsv", ".xlsx", ".xls") and not p.name.startswith("~$")])
if not files:
    print("No tabular spectra found. Upload CSV/XLSX files to", INPUT_DIR, "then rerun this cell.")
else:
    for i, f in enumerate(files): print(f"[{i}] {f.relative_to(INPUT_DIR)}")
FILE_INDEX = 0  # choose a listed file
if files:
    if FILE_INDEX < 0 or FILE_INDEX >= len(files): raise IndexError("FILE_INDEX is outside the discovered files list.")
    SELECTED_FILE = files[FILE_INDEX]
    if SELECTED_FILE.suffix.lower() in (".xlsx", ".xls"): df = pd.read_excel(SELECTED_FILE)
    else: df = pd.read_csv(SELECTED_FILE, sep="\t" if SELECTED_FILE.suffix.lower()==".tsv" else None, engine="python")
    df.columns = [str(c).strip() for c in df.columns]
    print("Rows/columns:", df.shape)
    display(df.head(3))
    print("Column names:", list(df.columns[:30]))

In [ ]:
# Flexible, inspectable wavelength detection for common spectra-table layouts.
# Do not guess wavelength positions for columns labelled Band_1, Band_2, etc.
def wavelength_nm(label):
    s = str(label).strip().lower().replace("μ", "u").replace("µ", "u")
    s = re.sub(r"\\s+", "", s)
    # Protect metadata; use full matches, not substring searches (e.g. "valid" contains "id").
    if re.fullmatch(r"(?:sample_?id|id|date|class|label|target|value|depth|site|group|index|unnamed:.*)", s):
        return None
    # e.g. 450, 450.5, X450, R_450, 450nm, reflectance_450, wl_450nm, band_450
    m = re.fullmatch(
        r"(?:x|r[_-]?|wl[_-]?|wavelength[_-]?|band[_-]?|reflectance[_-]?|refl[_-]?|"
        r"nm[_-]?|b[_-]?|lambda[_-]?)?(\\d{3,4}(?:[._]\\d+)?)"
        r"(?:nm)?", s
    )
    if not m:
        # e.g. "450 nm" already handled after removing whitespace
        return None
    value = float(m.group(1).replace("_", "."))
    return value if 350 <= value <= 2500 else None

def extract_spectra(table):
    """Return (waves, numeric spectral array, metadata, layout).
    Supports spectra in rows (wavelength headers) or spectra in columns
    (wavelengths in first/explicit column). No synthetic wavelength assumptions.
    """
    candidates = [(wavelength_nm(c), c) for c in table.columns]
    columns = [(float(w), c) for w, c in candidates if w is not None]
    if len(columns) >= 10:
        columns.sort(key=lambda t: t[0])
        waves = np.array([p[0] for p in columns], dtype=float)
        selected = [p[1] for p in columns]
        if len(np.unique(waves)) != len(waves):
            raise ValueError("Duplicate wavelength labels were found; resolve duplicates before processing.")
        matrix = table[selected].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=float)
        meta = table.drop(columns=selected).reset_index(drop=True)
        return waves, matrix, meta, "wavelength columns; samples in rows"
    # A common alternative has 'Wavelength (nm)' / 'nm' as a column and samples in other columns.
    preferred = [c for c in table.columns
                 if re.search(r"wavelength|^wl$|^nm$|^lambda", str(c), re.I)]
    possible = preferred + [c for c in table.columns[:2] if c not in preferred]
    for key in possible:
        wave_series = pd.to_numeric(table[key], errors="coerce")
        finite = wave_series.notna()
        w = wave_series[finite].to_numpy(dtype=float)
        if len(w) < 10 or not np.all((w >= 350) & (w <= 2500)) or len(np.unique(w)) != len(w):
            continue
        # Only accept a plausible wavelength axis (increasing or decreasing).
        if not (np.all(np.diff(w) > 0) or np.all(np.diff(w) < 0)):
            continue
        others = table.loc[finite, table.columns != key].apply(pd.to_numeric, errors="coerce")
        numeric_cols = [c for c in others if others[c].notna().sum() >= max(10, int(.8 * len(w)))]
        if not numeric_cols:
            continue
        order = np.argsort(w)
        matrix = others[numeric_cols].to_numpy(dtype=float)[order].T
        meta = pd.DataFrame({"spectrum_id": [str(c) for c in numeric_cols]})
        return w[order], matrix, meta, "wavelength rows; spectra in columns (transposed)"
    return None

if files:
    print("Selected file:", SELECTED_FILE.name, "| shape:", df.shape)
    print("First columns:", list(df.columns[:20]))
    parsed = extract_spectra(df)
    if parsed is None:
        print("\\nNo plausible wavelength axis detected in this selected file.")
        print("This may be a metadata file, a tabular file with multi-row headers,")
        print("or an image product (GeoTIFF/ENVI), not a spectral signature table.")
        print("Inspect FILE_INDEX and the sample preview above.")
        print("To use an imaging cube, a separate raster-specific workflow is required.")
        raise ValueError(
            "Unsupported spectral table layout. Please share the first 10 rows and column names "
            "of the selected file so we can adapt the importer without guessing wavelengths."
        )
    waves, spectra, metadata, layout = parsed
    cols = [f"{x:g}nm" for x in waves]
    print("Detected layout:", layout)
    print(f"{len(spectra)} spectra; {len(waves)} wavelengths; {waves.min():g}–{waves.max():g} nm")
    print("Non-finite cells:", int(np.size(spectra) - np.isfinite(spectra).sum()))
    if np.isfinite(spectra).any() and np.nanpercentile(spectra, 95) > 2:
        warnings.warn("Values may be percent reflectance or radiance. Confirm original units.")
    print("Metadata columns:", list(metadata.columns))


## 3. Quality control and spectral visualization
The default QC removes rows containing non-finite values. It flags potential out-of-range reflectance **without** altering measured values. Confirm whether the source is unitless reflectance, percent reflectance, or radiance before processing.

In [ ]:
if files:
    valid = np.isfinite(spectra).all(axis=1)
    X = spectra[valid]
    meta = metadata.loc[valid].reset_index(drop=True)
    qc = pd.DataFrame({"row":np.arange(len(spectra)), "complete":valid, "below_zero":np.any(spectra<0,axis=1), "above_one":np.any(spectra>1,axis=1)})
    qc.to_csv(OUTPUT_DIR/"spectral_qc.csv", index=False)
    print(f"Complete records: {valid.sum()}/{len(valid)}")
    print("Negative values flagged:", int(qc.below_zero.sum()), "Values >1 flagged:", int(qc.above_one.sum()))
    if len(X)==0: raise ValueError("No complete spectra. Investigate missing-data handling.")
    fig, ax=plt.subplots(figsize=(11,5))
    for row in X[:min(len(X),80)]: ax.plot(waves,row,lw=.7,alpha=.35)
    ax.plot(waves,np.median(X,axis=0),color="black",lw=2,label="Median")
    ax.set(xlabel="Wavelength (nm)",ylabel="Recorded spectral values",title="Measured spectral signatures")
    ax.legend(); fig.tight_layout(); fig.savefig(OUTPUT_DIR/"spectral_signatures.png",dpi=200); plt.show()

## 4. Compare classes when labels are available
Choose a categorical column, e.g. `landcover`, `class`, `material` or `sample_type`. Only plot class means when supplied labels exist. Differences alone do not establish causal or chemical interpretations.

In [ ]:
if files:
    likely = [c for c in meta.columns if c.lower() in ("class","label","landcover","material","sample_type","type","group")]
    CLASS_COLUMN = likely[0] if likely else None  # override if needed
    if CLASS_COLUMN:
        labels=meta[CLASS_COLUMN].fillna("Unknown").astype(str)
        fig,ax=plt.subplots(figsize=(11,5))
        for lab in labels.value_counts().index[:8]:
            z=X[(labels==lab).to_numpy()]
            ax.plot(waves,np.median(z,axis=0),label=f"{lab} (n={len(z)})")
        ax.set(xlabel="Wavelength (nm)",ylabel="Recorded spectral values",title="Median spectra by supplied sample class")
        ax.legend(fontsize=8);fig.tight_layout();fig.savefig(OUTPUT_DIR/"class_median_spectra.png",dpi=200);plt.show()
    else: print("No class column detected; set CLASS_COLUMN to a categorical metadata field to enable comparisons.")

## 5. Spectral preprocessing: Savitzky–Golay and SNV
Smooth/derive spectra only after checking wavelength spacing and observation units. SNV standardizes each spectrum separately. The first derivative is **not reflectance**; its units change. Avoid extrapolating across gaps in wavelength coverage.

In [ ]:
from scipy.signal import savgol_filter
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
if files:
    step=np.diff(waves)
    if not np.allclose(step,np.median(step),rtol=.05,atol=.2):
        print("Irregular wavelength spacing: SG derivative disabled. Inspect sensor bands and gaps.")
        X_deriv=None
    else:
        window=min(15, len(waves) if len(waves)%2==1 else len(waves)-1)
        if window < 5: raise ValueError("Insufficient bands for SG processing.")
        X_deriv=savgol_filter(X,window_length=window,polyorder=2,deriv=1,delta=float(np.median(step)),axis=1)
    stdev=X.std(axis=1,keepdims=True)
    keep=(stdev[:,0]>1e-12)
    if not keep.all():print("Zero-variance spectra omitted from SNV/PCA:",int((~keep).sum()))
    X_snv=(X[keep]-X[keep].mean(axis=1,keepdims=True))/stdev[keep]
    fig,ax=plt.subplots(figsize=(11,4))
    for row in X_snv[:min(70,len(X_snv))]: ax.plot(waves,row,alpha=.25,lw=.7)
    ax.plot(waves,np.median(X_snv,axis=0),color="black",lw=2,label="Median SNV")
    ax.set(xlabel="Wavelength (nm)",ylabel="SNV standardized value",title="Standard normal variate (SNV) spectra")
    ax.legend();fig.tight_layout();fig.savefig(OUTPUT_DIR/"spectra_snv.png",dpi=200);plt.show()
    if X_deriv is not None:
        pd.DataFrame(X_deriv,columns=[f"{w:g}nm" for w in waves]).to_csv(OUTPUT_DIR/"first_derivative.csv",index=False)

## 6. Principal component analysis (PCA)
Unsupervised PCA summarizes spectral variability; its axes do **not** identify soil properties, crop health or species. Examine outliers, loadings and explained variance before interpreting clusters.

In [ ]:
if files and len(X_snv)>=3:
    ncomp=min(3,len(X_snv)-1,X_snv.shape[1])
    model=PCA(n_components=ncomp).fit(X_snv)
    scores=model.transform(X_snv)
    fig,ax=plt.subplots(figsize=(7,5))
    ax.scatter(scores[:,0],scores[:,1],alpha=.7,s=24)
    ax.set(xlabel=f"PC1 ({model.explained_variance_ratio_[0]:.1%})",ylabel=f"PC2 ({model.explained_variance_ratio_[1]:.1%})",title="PCA of SNV spectra")
    fig.tight_layout();fig.savefig(OUTPUT_DIR/"pca_scores.png",dpi=200);plt.show()
    pd.DataFrame(scores,columns=[f"PC{i+1}" for i in range(ncomp)]).to_csv(OUTPUT_DIR/"pca_scores.csv",index=False)
else: print("PCA needs at least three complete non-constant spectra.")

## 7. Optional exercise — predict a measured target
Use this only when the source has an explicitly measured numeric target (such as laboratory SOC). Data must contain enough independent samples. This is a **demonstration**, not a publishable validation: random splitting is inappropriate for clustered field samples or replicates. Choose group-aware validation for real research.

In [ ]:
if files:
    TARGET_COLUMN = None  # e.g. "SOC" or "organic_matter"; must be a real measured column
    if TARGET_COLUMN is None:
        print("Prediction intentionally skipped. Set TARGET_COLUMN to a verified laboratory-measured numeric column.")
    else:
        from sklearn.model_selection import train_test_split
        from sklearn.cross_decomposition import PLSRegression
        from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
        if TARGET_COLUMN not in meta: raise KeyError(f"Missing target: {TARGET_COLUMN}")
        y=pd.to_numeric(meta.loc[keep,TARGET_COLUMN],errors="coerce").to_numpy()
        ok=np.isfinite(y)
        if ok.sum()<20: raise ValueError("At least 20 valid paired samples required for this classroom example.")
        Xa,Xb,ya,yb=train_test_split(X_snv[ok],y[ok],test_size=.25,random_state=42)
        # Fixed latent-variable count; no test-set tuning
        components=min(5,Xa.shape[0]-1,Xa.shape[1])
        reg=PLSRegression(n_components=components).fit(Xa,ya)
        pred=reg.predict(Xb).ravel()
        print({"n_test":len(yb),"R2":round(r2_score(yb,pred),3),"RMSE":round(np.sqrt(mean_squared_error(yb,pred)),3),"MAE":round(mean_absolute_error(yb,pred),3)})
        pd.DataFrame({"observed":yb,"predicted":pred}).to_csv(OUTPUT_DIR/"pls_holdout_predictions.csv",index=False)

## 8. Hyperspectral imaging interpretation and student questions
1. Which wavelength regions show the greatest spectral variability? Are any apparent features associated with known water, pigment or mineral absorptions?
2. How might soil moisture, atmospheric absorption and sensor signal-to-noise change your interpretation?
3. Why are point-level spectra not equivalent to satellite imaging spectroscopy pixels?
4. What information is lost or distorted by SNV and first-derivative preprocessing?
5. When would spatially independent validation be preferable to a random train/test split?

**Satellite extension:** Compare the concepts demonstrated here with imaging spectrometer products from NASA EMIT, EnMAP or PRISMA. Satellite scenes require geolocation, wavelength metadata, reflectance QA and spatial masking and are **not** interchangeable with the CSV spectral-table workflow.

**Deliverables:** your saved Colab notebook, spectral-signature and PCA figures, brief answers to the questions, and QC table. Check the exported files in your own Drive.

In [ ]:
print("Output folder:",OUTPUT_DIR)
for p in sorted(OUTPUT_DIR.glob("*")): print(" ",p.name, f"({p.stat().st_size:,} bytes)")